# Instructor Demo: APIs, HTTP, Async, and SDKs

**Audience:** Week 2 Day 3, before Activity 1 and Activity 2
**Environment:** VS Code with the Jupyter extension, Python 3.13, the shared repo-root `.venv`
**Estimated time:** 40 to 55 minutes (modular, skip sections the group already has)

## Coverage map

| This demo section | What it demos live before students build it |
|---|---|
| 1. Anatomy of a request/response | Activity 1, Section 1-2 |
| 2. Query parameters and pagination | Activity 1, Section 3 |
| 3. Rate limits and backoff, built up step by step | Activity 1, Section 4 |
| 4. requests vs httpx | Activity 1, Section 7 |
| 5. Sync vs async, built up step by step | Activity 2, Section 1-3 |
| 6. API vs SDK | Activity 2, Section 4 |

## Purpose

Show the full request/response/pagination/error-handling loop live, with a
different dataset than the one students use themselves (NYC Open Data instead
of GitHub/Open-Meteo), so this is not just a rehearsal of the exact lab. Every
piece of logic below is built up one small step at a time, the naive version
first, then the fix, the same way you would actually write and debug it.
Do not skip straight to the finished function; the point is watching it grow.

## Setup

1. Open this `Instructor_Demo/` folder in VS Code.
2. Open `API_Demo.ipynb`.
3. Select the repo-root `.venv` kernel (see `Activity_0_UV_API_Project_Setup.md`).
4. Run a cell to confirm the kernel works: `import requests; print("ok")`.

No API key is required for this demo. NYC Open Data (Socrata) needs no
signup at this call volume.

## 1. Anatomy of a request and a response

**PAUSE**: ask "what do you expect this URL to return, just from reading it?"
before running the next cell.

In [ ]:
import requests

url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
resp = requests.get(url, params={"$limit": 5}, timeout=10)
print("status:", resp.status_code)

**Expected output**: `status: 200`. We have not looked at the body yet, only
the receipt. That is deliberate: check the status before you touch the data.

In [ ]:
records = resp.json()
print("type:", type(records), "count:", len(records))
print("first record:", records[0])

**Expected observations**: the response is a JSON list of dict-like records,
the same shape as Day 2's list-of-dicts drills. Field names include
`unique_key`, `created_date`, `complaint_type`, and `borough`.

## 2. Fail loudly, then handle it

Show the crash first, unhandled, so the room sees what "loud" actually looks
like before you soften it.

In [ ]:
bad_url = "https://data.cityofnewyork.us/resource/not-a-real-dataset.json"
resp = requests.get(bad_url, timeout=10)
resp.raise_for_status()  # this line will raise; let it, and read the traceback together

**PAUSE**: read the traceback out loud. What exception class is it? What
does the message tell you? Now catch it instead of letting it kill the whole
script.

In [ ]:
try:
    resp = requests.get(bad_url, timeout=10)
    resp.raise_for_status()
except requests.exceptions.RequestException as exc:
    print(f"caught it: {exc}")

# PAUSE: what would happen to a real pipeline if this were `except: pass`
# instead of catching a specific exception and printing it?

## 3. Query parameters and pagination, built up

Start with the smallest possible version: one page.

In [ ]:
resp = requests.get(url, params={"$limit": 50, "$offset": 0}, timeout=10)
resp.raise_for_status()
page_one = resp.json()
print(f"page 1: {len(page_one)} records")

**PAUSE**: this dataset has thousands of complaints. `$limit: 50` capped us
at exactly 50. How would you get the *next* 50? (Answer: `$offset`.) Add one
more page by hand before writing any loop.

In [ ]:
resp = requests.get(url, params={"$limit": 50, "$offset": 50}, timeout=10)
resp.raise_for_status()
page_two = resp.json()
print(f"page 2: {len(page_two)} records")

all_records = page_one + page_two
print(f"combined so far: {len(all_records)} records")

Now generalize the pattern you just did by hand into a loop, so it works for
any number of pages instead of two copy-pasted cells.

In [ ]:
import time

all_records = []
for page in range(3):
    resp = requests.get(url, params={"$limit": 50, "$offset": page * 50}, timeout=10)
    resp.raise_for_status()
    page_records = resp.json()
    all_records.extend(page_records)
    print(f"page {page + 1}: {len(page_records)} records")
    time.sleep(0.3)

print(f"total: {len(all_records)} records")

**Expected output**

```text
page 1: 50 records
page 2: 50 records
page 3: 50 records
total: 150 records
```

**PAUSE**: how would you know when to stop paginating, if you did not already
decide "3 pages" up front? (Answer: stop when a page comes back with fewer
than `$limit` records; that is the signal you have reached the end.)

## 4. Rate limits and backoff, built up step by step

Do not write the finished retry function first. Build it up.

**Step 1**: what happens today, with no retry logic at all, if a call gets
rate limited? It just fails.

In [ ]:
def naive_get(url, params=None):
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()   # a 429 raises here and the caller has to deal with it
    return resp

resp = naive_get(url, params={"$limit": 5})
print("naive call succeeded:", resp.status_code)

**Step 2**: add the simplest possible retry, one fixed wait, no logic about
how many times to try.

In [ ]:
def get_with_one_retry(url, params=None):
    resp = requests.get(url, params=params, timeout=10)
    if resp.status_code == 429:
        print("  rate limited, waiting 1s and trying once more")
        time.sleep(1)
        resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    return resp

resp = get_with_one_retry(url, params={"$limit": 5})
print("succeeded:", resp.status_code)

**Step 3**: one retry is not enough for a real outage, and a fixed 1-second
wait can hammer a struggling server. Generalize to *several* tries, waiting
longer each time (exponential backoff).

In [ ]:
def polite_get(url, params=None, max_tries=3):
    """GET with exponential backoff on 429."""
    wait = 1
    for attempt in range(1, max_tries + 1):
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code != 429:
            resp.raise_for_status()
            return resp
        print(f"  attempt {attempt}: rate limited, waiting {wait}s")
        time.sleep(wait)
        wait *= 2
    raise RuntimeError("gave up after repeated rate limiting")

resp = polite_get(url, params={"$limit": 5})
print("succeeded:", resp.status_code)

**Expected output** for all three steps: since GitHub/Socrata is unlikely to
rate-limit one classroom demo, each should print its success line immediately
with no retry messages. The retry logic only fires under real 429 pressure,
but you build it in before you need it, not after a pipeline fails in
production. Point out explicitly: `polite_get` (step 3) is the shape students
will build again themselves, from scratch, in Activity 1.

## 5. `requests` versus `httpx`, live

Same call, both libraries, side by side.

In [ ]:
import httpx

r1 = requests.get(url, params={"$limit": 1}, timeout=10)
r1.raise_for_status()
print("requests:", r1.json()[0]["complaint_type"])

In [ ]:
with httpx.Client(timeout=10) as client:
    r2 = client.get(url, params={"$limit": 1})
    r2.raise_for_status()
    print("httpx:  ", r2.json()[0]["complaint_type"])

# PAUSE: name every line that is different between the two cells above.
# (Answer: almost none. That is the point.)

## 6. Sync versus async, built up step by step

Do not open with the finished `asyncio.gather` version. Build toward it.

**Step 1**: the sync version you already know, timed.

In [ ]:
offsets = [0, 50, 100, 150, 200]

start = time.perf_counter()
with httpx.Client(timeout=10) as client:
    for off in offsets:
        client.get(url, params={"$limit": 50, "$offset": off})
sync_seconds = time.perf_counter() - start
print(f"sync: {sync_seconds:.2f}s")

**Step 2**: write just the single-call coroutine first, on its own, before
worrying about running several of them together.

In [ ]:
async def fetch(client, off):
    return await client.get(url, params={"$limit": 50, "$offset": off})

# PAUSE: this cell defines a coroutine function but does not call anything yet.
# Nothing happens when you run this cell. That surprises people the first time.

**Step 3**: now run that coroutine several times *concurrently* with
`asyncio.gather`, and time it.

In [ ]:
import asyncio

async def run_async():
    async with httpx.AsyncClient(timeout=10) as client:
        return await asyncio.gather(*[fetch(client, off) for off in offsets])

start = time.perf_counter()
await run_async()
async_seconds = time.perf_counter() - start
print(f"async: {async_seconds:.2f}s")

## 7. API versus SDK, live

Quick contrast: raw HTTP to a NYC Open Data endpoint (an API with no SDK)
versus calling AWS through `boto3` (an SDK). This reads the same public,
anonymous-read S3 object (`s3://tscookbook/AirQualityUCI.csv`) students use in
Activity 2, so no AWS credentials are needed here either. We ask only for the
one object we want (`head_object` then `get_object`), not a bucket listing,
which the public object does not grant.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
BUCKET, KEY = "tscookbook", "AirQualityUCI.csv"

# metadata first (cheap), then the body
head = s3.head_object(Bucket=BUCKET, Key=KEY)
print(f"  {KEY}: {head['ContentType']}, {head['ContentLength']:,} bytes")

obj = s3.get_object(Bucket=BUCKET, Key=KEY)
first_line = obj["Body"].read().decode("utf-8").splitlines()[0]
print("  header row:", first_line[:60], "...")

**Talking point**: NYC Open Data has no official Python SDK, so raw
`requests`/`httpx` is the only option, which is fine for a public API. AWS
*could* be called the same raw way, but every authenticated request needs a
signed `Authorization` header (AWS Signature Version 4), which is tedious and
easy to get wrong by hand. `boto3` exists to do that signing for you. Today's
call is unsigned (a public object), so there is no signing to show, but the
client/method shape (`boto3.client(...)`, `.head_object(...)`,
`.get_object(...)`) is the same shape you would use for an authenticated call.
Note we never listed the bucket: a public object grants object reads, not
`s3:ListBucket`, and we already knew the key we wanted.

## Cleanup

No files are written by this demo; nothing to clean up.

## Troubleshooting

| Symptom | Likely cause | Quick fix |
|---|---|---|
| `ModuleNotFoundError` | Wrong kernel selected | Reselect the repo-root `.venv` kernel |
| Request hangs | No timeout or network issue | Keep `timeout=10`; retry on the classroom network |
| HTTP 429 during pagination | Whole classroom hitting Socrata at once | Lower `max_pages` or add `time.sleep(1)` between calls |
| `await` fails outside async function | Older Jupyter/ipykernel version | Confirm the kernel is the current `.venv`, not a stale one |
| `AccessDenied` / `NoSuchKey` on the S3 call | Bucket/key typo, or object no longer public | Confirm `s3://tscookbook/AirQualityUCI.csv` is reachable (curl it) before class |

## Backup plan

If NYC Open Data is unavailable, switch the dataset id from `erm2-nwe9` to
`h9gi-nx95`, or fall back to a hard-coded list of 3 dicts for the pagination
and async demos, and say so explicitly. Keep the error-handling and
sync-vs-async timing discussion either way; those are the load-bearing
concepts, not the specific dataset.